In [2]:
%pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [4]:
import pandas as pd
from sqlalchemy import create_engine
import geopandas as gpd
from shapely.geometry import Point
from dotenv import load_dotenv
import os

load_dotenv()

# Load GeoJSON data
schools_gdf = gpd.read_file("/home/dadams/Repos/colorado_spills/Public_School_Locations_-_Current.geojson")
nursing_homes_gdf = gpd.read_file("/home/dadams/Repos/colorado_spills/Nursing_Homes.geojson")

# Convert CRS to match oil spill dataset (if needed)
target_crs = "EPSG:4326"  # Adjust if necessary
schools_gdf = schools_gdf.to_crs(target_crs)
nursing_homes_gdf = nursing_homes_gdf.to_crs(target_crs)

# Database connection details from zshrc environment variables
db_name = 'colorado_spills'
user = os.getenv('DB_USER')
password = os.getenv('DB_PASSWORD')
host = os.getenv('DB_HOST')
port = os.getenv('DB_PORT', '5432')  # Ensure port is always set

# Create an engine to connect to the PostgreSQL database
engine = create_engine(f'postgresql+psycopg2://{user}:{password}@{host}:{port}/{db_name}')

# Query Longitude and Latitude correctly (case-sensitive)
spills_query = """
SELECT *, "Longitude", "Latitude" 
FROM spills_with_demographics_geog;
"""

# Load spills dataset from the correct table (fetch once for efficiency)
spills_df = pd.read_sql(spills_query, engine)

# Drop duplicate Longitude & Latitude columns if they exist
spills_df = spills_df.loc[:, ~spills_df.columns.duplicated()]

# Debugging step: Print unique column names to verify structure
print("Unique columns in spills_df:", spills_df.columns.tolist())

# Convert Longitude and Latitude to numeric
spills_df["Longitude"] = pd.to_numeric(spills_df["Longitude"], errors="coerce")
spills_df["Latitude"] = pd.to_numeric(spills_df["Latitude"], errors="coerce")

# Drop any remaining NaNs
spills_df = spills_df.dropna(subset=["Longitude", "Latitude"])

# Convert into GeoDataFrame
spills_gdf = gpd.GeoDataFrame(
    spills_df,
    geometry=gpd.points_from_xy(spills_df["Longitude"].values, spills_df["Latitude"].values)
)

# Ensure CRS is set properly (assuming it's WGS 84)
spills_gdf = spills_gdf.set_crs("EPSG:4326")

# Set buffer distance (adjust as needed)
buffer_distance = 5000  # 5 km radius

# Reproject to a projected CRS before buffering to avoid warnings
projected_crs = "EPSG:3857"  # Web Mercator Projection
schools_gdf = schools_gdf.to_crs(projected_crs)
nursing_homes_gdf = nursing_homes_gdf.to_crs(projected_crs)

# Apply buffering in projected CRS
schools_gdf["geometry"] = schools_gdf.geometry.buffer(buffer_distance)
nursing_homes_gdf["geometry"] = nursing_homes_gdf.geometry.buffer(buffer_distance)

# Convert back to geographic CRS
schools_gdf = schools_gdf.to_crs(target_crs)
nursing_homes_gdf = nursing_homes_gdf.to_crs(target_crs)

# Spatial join: Find spills near schools and nursing homes
spills_near_schools = gpd.sjoin(spills_gdf, schools_gdf, how="inner", predicate="intersects")
spills_near_nursing_homes = gpd.sjoin(spills_gdf, nursing_homes_gdf, how="inner", predicate="intersects")

# Aggregate statistics
school_spill_summary = spills_near_schools.groupby("NCESSCH").agg({
    "Oil Spill Volume": "sum",
    "Produced Water Spill Volume": "sum",
    "Report Delay (Days)": "mean"
}).reset_index()

nursing_home_spill_summary = spills_near_nursing_homes.groupby("ID").agg({
    "Oil Spill Volume": "sum",
    "Produced Water Spill Volume": "sum",
    "Report Delay (Days)": "mean"
}).reset_index()

# Save results
school_spill_summary.to_csv("school_spill_analysis.csv", index=False)
nursing_home_spill_summary.to_csv("nursing_home_spill_analysis.csv", index=False)

print("Analysis completed. Results saved to CSV files.")


Unique columns in spills_df: ['Document #', 'Report', 'Operator', 'Operator #', 'Tracking #', 'Initial Report Date', 'Date of Discovery', 'Spill Type', 'Qtr Qtr', 'Section', 'Township', 'range', 'meridian', 'Latitude', 'Longitude', 'Municipality', 'county', 'Facility Type', 'Facility ID', 'API County Code', 'API Sequence Number', 'Spilled outside of berms', 'More than five barrels spilled', 'Oil Spill Volume', 'Condensate Spill Volume', 'Flow Back Spill Volume', 'Produced Water Spill Volume', 'E&P Waste Spill Volume', 'Other Waste', 'Drilling Fluid Spill Volume', 'Current Land Use', 'Other Land Use', 'Weather Conditions', 'Surface Owner', 'Surface Owner Other', 'Waters of the State', 'Residence / Occupied Structure', 'livestock', 'Public Byway', 'Surface Water Supply Area', 'Spill Description', 'Supplemental Report Date', 'Oil BBLs Spilled', 'Oil BBLs Recovered', 'Oil Unknown', 'Condensate BBLs Spilled', 'Condensate BBLs Recovered', 'Condensate Unknown', 'Produced Water BBLs Spilled', 

In [3]:
# Debugging: Check column names in schools_gdf and nursing_homes_gdf
print("Columns in schools_gdf:", schools_gdf.columns.tolist())
print("Columns in nursing_homes_gdf:", nursing_homes_gdf.columns.tolist())

# Use the correct column name based on the dataset
correct_school_id_col = "school_id" if "school_id" in schools_gdf.columns else schools_gdf.columns[0]  # Choose first column if school_id is missing
correct_nursing_home_id_col = "nursing_home_id" if "nursing_home_id" in nursing_homes_gdf.columns else nursing_homes_gdf.columns[0]

# Aggregate statistics using correct column names
school_spill_summary = spills_near_schools.groupby(correct_school_id_col).agg({
    "Oil Spill Volume": "sum",
    "Produced Water Spill Volume": "sum",
    "Report Delay (Days)": "mean"
}).reset_index()

nursing_home_spill_summary = spills_near_nursing_homes.groupby(correct_nursing_home_id_col).agg({
    "Oil Spill Volume": "sum",
    "Produced Water Spill Volume": "sum",
    "Report Delay (Days)": "mean"
}).reset_index()


Columns in schools_gdf: ['OBJECTID', 'NCESSCH', 'LEAID', 'NAME', 'OPSTFIPS', 'STREET', 'CITY', 'STATE', 'ZIP', 'STFIP', 'CNTY', 'NMCNTY', 'LOCALE', 'LAT', 'LON', 'CBSA', 'NMCBSA', 'CBSATYPE', 'CSA', 'NMCSA', 'NECTA', 'NMNECTA', 'CD', 'SLDL', 'SLDU', 'SCHOOLYEAR', 'geometry']
Columns in nursing_homes_gdf: ['OBJECTID', 'ID', 'NAME', 'ADDRESS', 'CITY', 'STATE', 'ZIP', 'ZIP4', 'TELEPHONE', 'TYPE', 'STATUS', 'POPULATION', 'COUNTY', 'COUNTYFIPS', 'COUNTRY', 'LATITUDE', 'LONGITUDE', 'NAICS_CODE', 'NAICS_DESC', 'SOURCE', 'SOURCEDATE', 'VAL_METHOD', 'VAL_DATE', 'WEBSITE', 'TOT_RES', 'TOT_STAFF', 'BEDS', 'EXCESS_BED', 'OWNERSHIP', 'MEDICAIDID', 'MEDICAREID', 'STATE_LIC', 'SOURCETYPE', 'geometry']
